In [1]:
import os

In [2]:
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model\\research'

In [4]:
# os.chdir("..")

In [5]:
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model'

In [9]:
import pandas as pd
import joblib
import numpy as np
from src.customer_churn.logging.logger import logging
from src.customer_churn.exception.exception import CustomerChurnException
from src.customer_churn.utils.main_utils.common import read_yaml_file
import sys


# Path initialization:

In [10]:
pre_processing_artifacts = "Artifacts/pre_processing/pre_processor.pkl"
model_trainer_artifacst = "Artifacts/model_training/final_model.pkl"

# Create the prediction pipeline:

In [11]:
class Prediction:
    
    def __init__(self):
        try:
            self.pre_processor = joblib.load(pre_processing_artifacts)
            self.model = joblib.load(model_trainer_artifacst)
            logging.info("Prediction class initialization success!")
        except Exception as e:
            raise CustomerChurnException(e, sys)


    def predict(self, X_features) -> dict:
        """
        Takes in the features and predicts on it.

        Returns:
        ---------
        results: a dictionary consisiting of the predicted result, predictied probability, and risk label
        """
        try:
            results = []
            X_features_transformed = self.pre_processor.transform(X_features)
            logging.info("Features transformation success!")
            for pred_prob, prediction in zip(self.model.predict_proba(X_features_transformed), self.model.predict(X_features_transformed)):
                prediction_probability = round(float(pred_prob[1]), 4)
                if prediction == 1:
                    if prediction_probability < 0.65:
                        risk_label = 'Low Risk'
                    elif prediction_probability < 0.80:
                        risk_label = 'Medium Risk'
                    else:
                        risk_label = 'High Risk'
                else:
                    risk_label = 'No Risk'
                results.append({
                    'prediction_probability': prediction_probability,
                    'prediction': 'Churn'if prediction == 1 else 'Stay',
                    'risk_label': risk_label
                })
            return results
        except Exception as e:
            raise CustomerChurnException(e, sys)
        

In [12]:
test = pd.read_parquet("Artifacts/pre_processing/full_dataset/X.parquet")

In [13]:
test.columns

Index(['order_frequency', 'total_monetary_value', 'total_quantity_abs',
       'total_order_issues', 'country', 'avg_order_value',
       'avg_quantity_per_order', 'customer_order_issue_rate', 'recency',
       'tenure', 'avg_stockcode_issue_rate', 'max_stockcode_issue_rate',
       'total_orders_made_for_stock'],
      dtype='object')

In [14]:
features = test.tail(5)

In [15]:
features

,order_frequency,total_monetary_value,total_quantity_abs,total_order_issues,country,avg_order_value,avg_quantity_per_order,customer_order_issue_rate,recency,tenure,avg_stockcode_issue_rate,max_stockcode_issue_rate,total_orders_made_for_stock
8494,6,146.60,74,0,United Kingdom,24.433333,12.333333,0.0,128,404,0.021818,0.119578,20664.0
8495,1,70.68,84,0,United Kingdom,70.680000,84.000000,0.0,177,177,0.017649,0.040816,547.0
8496,1,83.90,31,0,United Kingdom,83.900000,31.000000,0.0,406,406,0.033175,0.050000,820.0
8497,2,192.52,144,0,United Kingdom,96.260000,72.000000,0.0,222,469,0.016174,0.038462,2866.0
8498,3,605.12,474,0,United Kingdom,201.706667,158.000000,0.0,128,317,0.031735,0.108392,5383.0


In [16]:
pred = Prediction()
result = pred.predict(X_features=features)

In [17]:
result

[{'prediction_probability': 0.4409,
  'prediction': 'Stay',
  'risk_label': 'No Risk'},
 {'prediction_probability': 0.7549,
  'prediction': 'Churn',
  'risk_label': 'Medium Risk'},
 {'prediction_probability': 0.7669,
  'prediction': 'Churn',
  'risk_label': 'Medium Risk'},
 {'prediction_probability': 0.7221,
  'prediction': 'Churn',
  'risk_label': 'Medium Risk'},
 {'prediction_probability': 0.6489,
  'prediction': 'Churn',
  'risk_label': 'Low Risk'}]